# UCS3564 – Natural Language Processing
## Assignment 2: Concept Extraction using PoS and Dependency Parsing
Toolkit: **Stanza** (PoS + dependency parsing) with an **NLTK** regex-chunker baseline for comparison.

In [ ]:
# Setup: run once
!pip install -q stanza nltk networkx matplotlib
import stanza, nltk
stanza.download("en", verbose=False)
nltk.download("punkt", quiet=True); nltk.download("punkt_tab", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

In [ ]:
import stanza
nlp = stanza.Pipeline("en", processors="tokenize,pos,lemma,depparse", verbose=False)

text = (
    "A bicycle is a simple machine that people use for transport, exercise, and fun. "
    "It has a light metal frame, thin rubber tires, and a strong steel chain that turns the back wheel. "
    "Many cities now offer public bike sharing programs, free bike repair stands, and safe bike lanes "
    "to make daily cycling easier and safer for workers and students. "
    "Some modern bikes come with extra features such as multi speed gear systems, disc brakes, "
    "and small built in lights for night riding."
)
doc = nlp(text)

# Inspect the parse (PoS + dependency) for every word
for s_idx, sent in enumerate(doc.sentences, 1):
    print(f"--- Sentence {s_idx} ---")
    for w in sent.words:
        head = sent.words[w.head - 1].text if w.head > 0 else "ROOT"
        print(f"{w.id:>3} {w.text:<12} {w.upos:<6} {w.xpos:<5} {w.deprel:<14} -> {head}")

## Part 1 – Rule-based concept extraction

Rules (dependency + PoS):
1. **Heads**: any word with `upos` in `NOUN`, `PROPN` (and gerund nouns such as *cycling*, *riding* tagged `NOUN`/`VERB` with `deprel` that makes them a noun phrase head – handled via the pattern below).
2. **Left modifiers**: dependents of the head with `deprel` in `amod`, `compound`, `nummod`, `flat` are included recursively, so **stacked adjectives** and **compound noun chains** are captured.
3. **Coordination**: every noun linked by `conj` to a noun head is a head itself, and gets its *own* modifiers (`light metal frame`, `thin rubber tires`, `strong steel chain` are separate concepts).
4. **Contiguity repair**: parser quirks such as `built in lights` or `multi speed` are repaired by filling gaps between the leftmost modifier and the head when the gap tokens are not punctuation, determiners or conjunctions.
5. **Stop-modifiers** such as quantifiers (*many, some*) are removed.
6. **Prepositional attachment**: for a head with an `nmod`/`obl` child introduced by a `case` word (*of, for, with*), the **pobj phrase** is extracted as a concept and the extended phrase `head + prep + phrase` is stored as an additional concept.

In [ ]:
MOD_RELS = {"amod", "compound", "nummod", "flat", "compound:prt"}
HEAD_UPOS = {"NOUN", "PROPN"}
SKIP_WORDS = {"many", "some", "more", "most", "much", "any", "other", "several"}
GAP_BLOCK_UPOS = {"PUNCT", "DET", "CCONJ", "SCONJ", "PRON"}


def collect_modifiers(sent, head):
    """Recursively collect ids of modifier words attached to `head`."""
    ids = set()
    for w in sent.words:
        if w.head == head.id and w.deprel in MOD_RELS and w.text.lower() not in SKIP_WORDS:
            ids.add(w.id)
            ids |= collect_modifiers(sent, w)              # nested modifiers
            for sub in sent.words:                          # e.g. 'built' + 'in' / 'multi' + 'speed'
                if sub.head == w.id and sub.deprel in {"advmod", "compound:prt", "compound"}:
                    ids.add(sub.id)
    return ids


def build_phrase(sent, head):
    """Return the contiguous noun phrase text built around `head`."""
    ids = collect_modifiers(sent, head) | {head.id}
    lo = min(ids)
    # contiguity repair: fill harmless gaps left of the head
    for wid in range(lo, head.id):
        w = sent.words[wid - 1]
        if w.upos in GAP_BLOCK_UPOS:
            ids = {i for i in ids if i > wid}
            lo = min(ids)
    for wid in range(lo, head.id):
        ids.add(wid)
    return " ".join(sent.words[i - 1].text.lower() for i in sorted(ids))


def is_np_head(word):
    return word.upos in HEAD_UPOS


def extract_concepts(sent):
    """Return list of (concept, rule_used) for one sentence."""
    found = []
    for w in sent.words:
        if not is_np_head(w):
            continue
        phrase = build_phrase(sent, w)
        rule = "head noun"
        if w.deprel == "conj":
            rule = "conj-coordinated noun (own modifiers)"
        elif any(sent.words[i - 1].deprel == "compound" for i in range(1, len(sent.words) + 1)
                 if sent.words[i - 1].head == w.id):
            rule = "compound chain (+amod)" if " " in phrase else rule
        elif " " in phrase:
            rule = "amod/nummod modifiers + head"
        found.append((phrase, rule))
    return found


def extract_pp_concepts(sent):
    """Prepositional attachment: head + case + nmod/obl."""
    found = []
    for w in sent.words:
        if w.deprel in {"nmod", "obl"} and w.upos in HEAD_UPOS:
            cases = [c for c in sent.words if c.head == w.id and c.deprel == "case"]
            if not cases or w.head == 0:
                continue
            head = sent.words[w.head - 1]
            if head.upos not in HEAD_UPOS:
                continue
            prep = " ".join(c.text.lower() for c in sorted(cases, key=lambda x: x.id))
            found.append((f"{build_phrase(sent, head)} {prep} {build_phrase(sent, w)}",
                          "PP attachment (head + case + nmod)"))
    return found


all_concepts = []   # (sentence_idx, concept, rule)
for s_idx, sent in enumerate(doc.sentences, 1):
    for concept, rule in extract_concepts(sent) + extract_pp_concepts(sent):
        all_concepts.append((s_idx, concept, rule))

print("Extracted concepts (unique):")
seen = set()
for s_idx, concept, rule in all_concepts:
    if concept not in seen:
        seen.add(concept)
        print(f"  S{s_idx}: {concept:<45} <- {rule}")

In [ ]:
# Compare with the sample expected output
expected = ["bicycle", "simple machine", "light metal frame", "thin rubber tires", "strong steel chain",
            "back wheel", "public bike sharing programs", "free bike repair stands", "safe bike lanes",
            "daily cycling", "multi speed gear systems", "disc brakes", "built in lights", "night riding"]
extracted = {c for _, c, _ in all_concepts}
print("Expected concepts found:")
for e in expected:
    print(f"  [{'OK' if e in extracted else 'MISSING'}] {e}")

## Part 2 – Additional concepts and why a naive rule fails

A naive `JJ? NN+` chunker (below) is the baseline. Run it and compare against the dependency rules.

In [ ]:
import nltk
grammar = "NP: {<JJ>*<NN.*>+}"          # naive: adjectives followed by nouns
chunker = nltk.RegexpParser(grammar)

naive = []
for sent in nltk.sent_tokenize(text):
    tree = chunker.parse(nltk.pos_tag(nltk.word_tokenize(sent)))
    for st in tree.subtrees(lambda t: t.label() == "NP"):
        naive.append(" ".join(w.lower() for w, _ in st.leaves()))

naive_set = set(naive)
print("Naive chunker found:", sorted(naive_set))
print()
print("Found by dependency rules but missed/altered by naive rule:")
for s_idx, concept, rule in all_concepts:
    if concept not in naive_set and concept not in {x for x in ()}:
        print(f"  S{s_idx}: {concept:<45} rule: {rule}")

### Explanation table (fill / edit after reading your output)

| Additional concept type | Example from the passage | PoS / dependency used | Why naive `ADJ + NOUN` fails |
|---|---|---|---|
| Coordinated NPs | *free bike repair stands*, *safe bike lanes*, *disc brakes* | `conj` links nouns; each conjunct keeps its own `amod`/`compound` | Naive rule has no notion of coordination; it can merge or drop conjuncts after commas/"and". |
| Compound noun chains | *bike sharing programs*, *multi speed gear systems* | `compound` (NOUN/VERB + NOUN) | *sharing* is VBG and *multi* is not JJ, so `JJ* NN+` breaks. |
| Stacked adjectives | *light metal frame*, *thin rubber tires* | repeated `amod` + `compound` | Single-adjective rules stop after one modifier. |
| Gerund concepts | *daily cycling*, *night riding* | `amod`/`compound` on VBG tagged noun | Gerunds are often tagged VBG, not NN. |
| Prepositional attachment | *features with …*, *cycling for workers and students*, *programs for …* | `nmod`/`obl` + `case` (*for, of, with*) | A flat chunker never crosses the preposition, so the relation is lost. |
| Coordinated objects of prepositions | *workers*, *students* | `conj` under `nmod` with `case` | Needs the head–conjunct link, not adjacency. |

---
## Additional Task (Bonus) – Minimal Knowledge Graph

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

concept_set = {c for _, c, _ in all_concepts}


def concept_of(sent, word):
    """Map a noun word to the concept phrase containing it (longest match)."""
    if word.upos not in HEAD_UPOS:
        return None
    return build_phrase(sent, word)


graph = nx.DiGraph()
for sent in doc.sentences:
    for verb in sent.words:
        if verb.upos not in {"VERB", "AUX"}:
            continue
        subjects = [w for w in sent.words if w.head == verb.id and w.deprel in {"nsubj", "nsubj:pass"}]
        objects = [w for w in sent.words if w.head == verb.id and w.deprel in {"obj", "obl", "xcomp"}]
        # `be` copula: subject --is--> predicate noun
        if verb.upos == "AUX" and verb.deprel == "cop":
            subjects = [w for w in sent.words if w.head == verb.head and w.deprel == "nsubj"]
            objects = [sent.words[verb.head - 1]]
        for subj in subjects:
            for obj in objects:
                s_c, o_c = concept_of(sent, subj), concept_of(sent, obj)
                if s_c and o_c:
                    graph.add_edge(s_c, o_c, label=verb.lemma)
                # coordinated objects
                for conj in [w for w in sent.words if w.head == obj.id and w.deprel == "conj"]:
                    c_c = concept_of(sent, conj)
                    if s_c and c_c:
                        graph.add_edge(s_c, c_c, label=verb.lemma)

# nmod links (e.g. 'bike lanes' -for-> 'workers')
for sent in doc.sentences:
    for w in sent.words:
        if w.deprel == "nmod" and w.upos in HEAD_UPOS and w.head > 0:
            head = sent.words[w.head - 1]
            cases = [c.text.lower() for c in sent.words if c.head == w.id and c.deprel == "case"]
            if head.upos in HEAD_UPOS and cases:
                graph.add_edge(concept_of(sent, head), concept_of(sent, w), label=" ".join(cases))

print("Nodes:", graph.number_of_nodes(), "| Edges:", graph.number_of_edges())
for u, v, d in graph.edges(data=True):
    print(f"  {u}  --[{d['label']}]-->  {v}")

plt.figure(figsize=(14, 9))
pos = nx.spring_layout(graph, k=0.9, seed=42)
nx.draw(graph, pos, with_labels=True, node_color="#cfe8ff", node_size=2200, font_size=8, arrowsize=15)
nx.draw_networkx_edge_labels(graph, pos, edge_labels=nx.get_edge_attributes(graph, "label"), font_size=8)
plt.title("Minimal Knowledge Graph")
plt.show()

**Comment (2–3 lines):** The graph captures subject–verb–object relations such as
*people –use→ simple machine / transport* and *cities –offer→ bike lanes*, which a flat noun-phrase list
cannot show. A list only says that these concepts exist; the graph tells us how they are related
(who uses or offers what, and which feature belongs to which object).